# mBERT merged-seven classifier

This standalone notebook downloads the public model repository:

`hannah-khallaf/e2r-mbert-merged7`

Selected seed: **13**

A GPU is recommended but mBERT can also run on CPU.

No Hugging Face login or AIRE path is required.

The loader first checks for a standard Transformers or PEFT release. If the
repository contains only `best_model.pt`, it uses a compatibility loader that
reconstructs the expected classification model. The notebook stops rather than
returning unreliable predictions when the trained classification head cannot
be loaded.

In [ ]:
%pip install -q \
  "torch" \
  "transformers>=4.45,<5" \
  "huggingface_hub>=0.36,<1" \
  "peft>=0.13" \
  "accelerate>=1.0" \
  "bitsandbytes>=0.44" \
  "safetensors>=0.4" \
  "pandas>=2.0" \
  "pyyaml>=6.0"

In [ ]:
REPO_ID = "hannah-khallaf/e2r-mbert-merged7"
BASE_MODEL = "google-bert/bert-base-multilingual-cased"
SELECTED_SEED = 13
IS_QLORA = False
MAX_LENGTH = 256

print("Repository:", REPO_ID)
print("Selected seed:", SELECTED_SEED)

In [ ]:

from __future__ import annotations

import gc
import json
import re
import warnings
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml
from huggingface_hub import list_repo_files, snapshot_download
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
)

try:
    from peft import (
        LoraConfig,
        PeftModel,
        TaskType,
        get_peft_model,
    )
except ImportError:
    LoraConfig = PeftModel = TaskType = get_peft_model = None


LABELS = [
    "Compression",
    "Explanation",
    "Illocutionary Change",
    "Modulation",
    "Omission",
    "Synonymy",
    "Syntactic Change",
]


def repo_snapshot() -> Path:
    files = list_repo_files(REPO_ID, repo_type="model", token=False)
    print(f"Public repository files ({len(files)}):")
    for name in files:
        print(" ", name)

    return Path(
        snapshot_download(
            repo_id=REPO_ID,
            repo_type="model",
            token=False,
        )
    )


def find_one(root: Path, names: list[str]) -> Path | None:
    for name in names:
        direct = root / name
        if direct.is_file():
            return direct
        matches = sorted(root.rglob(name))
        if matches:
            return matches[0]
    return None


def has_full_hf_model(root: Path) -> bool:
    config = find_one(root, ["config.json"])
    weights = (
        list(root.rglob("model*.safetensors"))
        + list(root.rglob("pytorch_model*.bin"))
    )
    return config is not None and bool(weights)


def has_peft_adapter(root: Path) -> bool:
    config = find_one(root, ["adapter_config.json"])
    weights = (
        list(root.rglob("adapter_model*.safetensors"))
        + list(root.rglob("adapter_model*.bin"))
    )
    return config is not None and bool(weights)


def find_portable_directory(root: Path, adapter: bool) -> Path:
    config_name = "adapter_config.json" if adapter else "config.json"
    config = find_one(root, [config_name])
    if config is None:
        raise FileNotFoundError(config_name)
    return config.parent


def load_torch_checkpoint(path: Path) -> tuple[dict[str, torch.Tensor], dict[str, Any]]:
    try:
        obj = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        obj = torch.load(path, map_location="cpu")

    metadata: dict[str, Any] = {}

    if isinstance(obj, dict):
        metadata = {
            key: value
            for key, value in obj.items()
            if not isinstance(value, torch.Tensor)
            and key not in {"state_dict", "model_state_dict", "model"}
        }

        for key in ("model_state_dict", "state_dict", "model"):
            value = obj.get(key)
            if isinstance(value, dict) and any(
                isinstance(item, torch.Tensor)
                for item in value.values()
            ):
                return {
                    name: tensor
                    for name, tensor in value.items()
                    if isinstance(tensor, torch.Tensor)
                }, metadata

        if any(isinstance(value, torch.Tensor) for value in obj.values()):
            return {
                name: tensor
                for name, tensor in obj.items()
                if isinstance(tensor, torch.Tensor)
            }, metadata

    raise RuntimeError(
        f"Could not identify a state dictionary inside {path}."
    )


def recursively_find(config: Any, candidate_keys: set[str]) -> Any:
    if isinstance(config, dict):
        for key, value in config.items():
            if str(key).lower() in candidate_keys:
                return value
        for value in config.values():
            result = recursively_find(value, candidate_keys)
            if result is not None:
                return result
    elif isinstance(config, list):
        for value in config:
            result = recursively_find(value, candidate_keys)
            if result is not None:
                return result
    return None


def load_yaml_configs(root: Path) -> dict[str, Any]:
    merged: dict[str, Any] = {}
    for path in sorted(root.rglob("*.yaml")) + sorted(root.rglob("*.yml")):
        try:
            with path.open("r", encoding="utf-8") as handle:
                value = yaml.safe_load(handle)
            if isinstance(value, dict):
                merged[path.name] = value
        except Exception:
            continue
    return merged


def strip_prefix(state: dict[str, torch.Tensor], prefix: str) -> dict[str, torch.Tensor]:
    return {
        (key[len(prefix):] if key.startswith(prefix) else key): value
        for key, value in state.items()
    }


def state_candidates(state: dict[str, torch.Tensor]) -> list[dict[str, torch.Tensor]]:
    candidates = [state]
    prefixes = [
        "module.",
        "model.",
        "module.model.",
        "_orig_mod.",
        "network.",
    ]
    for prefix in prefixes:
        candidates.append(strip_prefix(state, prefix))

    # Some custom wrappers call the backbone "encoder" or "backbone".
    for prefix in ("encoder.", "backbone."):
        stripped = strip_prefix(state, prefix)
        candidates.append(stripped)

        bert_prefixed = {}
        for key, value in stripped.items():
            if key.startswith(
                ("embeddings.", "encoder.", "pooler.")
            ):
                bert_prefixed[f"bert.{key}"] = value
            else:
                bert_prefixed[key] = value
        candidates.append(bert_prefixed)

    unique = []
    seen = set()
    for candidate in candidates:
        signature = tuple(sorted(candidate.keys())[:20])
        if signature not in seen:
            seen.add(signature)
            unique.append(candidate)
    return unique


def choose_best_state(
    model: torch.nn.Module,
    state: dict[str, torch.Tensor],
) -> tuple[dict[str, torch.Tensor], int]:
    model_state = model.state_dict()
    best = state
    best_matches = -1

    for candidate in state_candidates(state):
        matches = sum(
            key in model_state
            and tuple(model_state[key].shape) == tuple(value.shape)
            for key, value in candidate.items()
        )
        if matches > best_matches:
            best = candidate
            best_matches = matches

    return best, best_matches


def classification_head_loaded(
    model: torch.nn.Module,
    loaded_state: dict[str, torch.Tensor],
) -> bool:
    model_keys = set(model.state_dict())
    loaded_keys = set(loaded_state)
    head_terms = ("classifier", "score", "classification_head")

    expected_head = {
        key for key in model_keys
        if any(term in key.lower() for term in head_terms)
    }
    loaded_head = expected_head & loaded_keys

    print("Classification-head keys expected:", len(expected_head))
    print("Classification-head keys loaded:", len(loaded_head))
    return bool(loaded_head)


def make_quantization_config() -> BitsAndBytesConfig:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "A CUDA GPU is required for the 7B QLoRA checkpoint."
        )

    dtype = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=dtype,
    )


def infer_target_modules(state: dict[str, torch.Tensor]) -> list[str]:
    modules = set()
    for key in state:
        match = re.search(r"\.([^.]+)\.lora_[AB](?:\.|$)", key)
        if match:
            modules.add(match.group(1))
    return sorted(modules)


def infer_modules_to_save(state: dict[str, torch.Tensor]) -> list[str] | None:
    modules = set()
    for key in state:
        match = re.search(r"\.([^.]+)\.modules_to_save\.", key)
        if match:
            modules.add(match.group(1))
    if modules:
        return sorted(modules)

    for candidate in ("score", "classifier"):
        if any(candidate in key.lower() for key in state):
            modules.add(candidate)
    return sorted(modules) or None


def infer_lora_rank(state: dict[str, torch.Tensor]) -> int | None:
    for key, tensor in state.items():
        if "lora_a" in key.lower() and tensor.ndim == 2:
            return int(tensor.shape[0])
    return None


def load_model_from_public_repo(root: Path):
    id2label = {index: label for index, label in enumerate(LABELS)}
    label2id = {label: index for index, label in enumerate(LABELS)}

    # Preferred case: the repository is already portable.
    if has_peft_adapter(root):
        if PeftModel is None:
            raise ImportError("Install peft before loading this adapter.")

        adapter_dir = find_portable_directory(root, adapter=True)
        with (adapter_dir / "adapter_config.json").open(
            "r", encoding="utf-8"
        ) as handle:
            adapter_config = json.load(handle)

        base_name = adapter_config.get(
            "base_model_name_or_path", BASE_MODEL
        )
        tokenizer = AutoTokenizer.from_pretrained(
            adapter_dir if (adapter_dir / "tokenizer_config.json").exists()
            else base_name
        )

        config = AutoConfig.from_pretrained(
            base_name,
            num_labels=len(LABELS),
            id2label=id2label,
            label2id=label2id,
            problem_type="multi_label_classification",
        )
        base = AutoModelForSequenceClassification.from_pretrained(
            base_name,
            config=config,
            quantization_config=make_quantization_config()
            if IS_QLORA else None,
            device_map="auto" if torch.cuda.is_available() else None,
            ignore_mismatched_sizes=True,
        )
        model = PeftModel.from_pretrained(base, adapter_dir)
        return model, tokenizer, "portable PEFT adapter"

    if has_full_hf_model(root):
        model_dir = find_portable_directory(root, adapter=False)
        tokenizer = AutoTokenizer.from_pretrained(
            model_dir if (model_dir / "tokenizer_config.json").exists()
            else BASE_MODEL
        )
        model = AutoModelForSequenceClassification.from_pretrained(
            model_dir
        )
        if torch.cuda.is_available():
            model = model.to("cuda")
        return model, tokenizer, "portable Transformers model"

    # Compatibility case: reconstruct the model from best_model.pt.
    checkpoint_path = find_one(root, ["best_model.pt"])
    if checkpoint_path is None:
        raise FileNotFoundError(
            "No portable model files or best_model.pt were found."
        )

    state, checkpoint_metadata = load_torch_checkpoint(checkpoint_path)
    print("Raw checkpoint tensors:", len(state))
    print("Checkpoint metadata keys:", sorted(checkpoint_metadata)[:30])

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    config = AutoConfig.from_pretrained(
        BASE_MODEL,
        num_labels=len(LABELS),
        id2label=id2label,
        label2id=label2id,
        problem_type="multi_label_classification",
    )

    if not IS_QLORA:
        model = AutoModelForSequenceClassification.from_pretrained(
            BASE_MODEL,
            config=config,
            ignore_mismatched_sizes=True,
        )
        selected_state, matches = choose_best_state(model, state)
        result = model.load_state_dict(selected_state, strict=False)
        print("Matched tensors:", matches)
        print("Missing keys:", len(result.missing_keys))
        print("Unexpected keys:", len(result.unexpected_keys))

        if not classification_head_loaded(model, selected_state):
            raise RuntimeError(
                "The raw checkpoint did not load a classification head. "
                "The original project model class is required."
            )

        model = model.to("cuda" if torch.cuda.is_available() else "cpu")
        return model, tokenizer, "raw checkpoint compatibility loader"

    if get_peft_model is None:
        raise ImportError("Install peft before loading this QLoRA checkpoint.")

    configs = load_yaml_configs(root)
    rank = recursively_find(configs, {"r", "lora_r", "rank"})
    alpha = recursively_find(
        configs, {"lora_alpha", "alpha"}
    )
    dropout = recursively_find(
        configs, {"lora_dropout", "adapter_dropout"}
    )
    target_modules = recursively_find(
        configs, {"target_modules", "lora_target_modules"}
    )
    modules_to_save = recursively_find(
        configs, {"modules_to_save"}
    )

    rank = int(rank) if rank is not None else infer_lora_rank(state)
    if rank is None:
        raise RuntimeError(
            "Could not infer the LoRA rank from the config or checkpoint."
        )

    alpha = float(alpha) if alpha is not None else float(2 * rank)
    dropout = float(dropout) if dropout is not None else 0.0

    if isinstance(target_modules, str):
        target_modules = [
            item.strip()
            for item in target_modules.split(",")
            if item.strip()
        ]
    if not target_modules:
        target_modules = infer_target_modules(state)
    if not target_modules:
        raise RuntimeError(
            "Could not infer LoRA target modules."
        )

    if isinstance(modules_to_save, str):
        modules_to_save = [
            item.strip()
            for item in modules_to_save.split(",")
            if item.strip()
        ]
    if not modules_to_save:
        modules_to_save = infer_modules_to_save(state)

    print("LoRA rank:", rank)
    print("LoRA alpha:", alpha)
    print("LoRA dropout:", dropout)
    print("Target modules:", target_modules)
    print("Modules to save:", modules_to_save)

    base = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL,
        config=config,
        quantization_config=make_quantization_config(),
        device_map="auto",
        ignore_mismatched_sizes=True,
    )

    lora_config = LoraConfig(
        r=rank,
        lora_alpha=alpha,
        lora_dropout=dropout,
        target_modules=target_modules,
        modules_to_save=modules_to_save,
        bias="none",
        task_type=TaskType.SEQ_CLS,
    )
    model = get_peft_model(base, lora_config)

    selected_state, matches = choose_best_state(model, state)
    result = model.load_state_dict(selected_state, strict=False)
    print("Matched tensors:", matches)
    print("Missing keys:", len(result.missing_keys))
    print("Unexpected keys:", len(result.unexpected_keys))

    if not classification_head_loaded(model, selected_state):
        raise RuntimeError(
            "The raw QLoRA checkpoint did not load the trained "
            "classification head. Export it with modules_to_save before "
            "presenting the repository as an inference-ready model."
        )

    return model, tokenizer, "raw QLoRA checkpoint compatibility loader"


def normalise_label(value: str) -> str:
    return value.replace("_", " ").strip().lower()


def extract_threshold_vector(value: Any) -> np.ndarray | None:
    if isinstance(value, list) and len(value) == len(LABELS):
        try:
            return np.asarray(value, dtype=float)
        except (TypeError, ValueError):
            return None

    if isinstance(value, dict):
        mapping = {
            normalise_label(str(key)): item
            for key, item in value.items()
        }
        if all(normalise_label(label) in mapping for label in LABELS):
            try:
                return np.asarray(
                    [
                        mapping[normalise_label(label)]
                        for label in LABELS
                    ],
                    dtype=float,
                )
            except (TypeError, ValueError):
                pass

        for key in (
            "thresholds",
            "best_thresholds",
            "per_label_thresholds",
            "decision_thresholds",
        ):
            if key in value:
                result = extract_threshold_vector(value[key])
                if result is not None:
                    return result

        for nested in value.values():
            result = extract_threshold_vector(nested)
            if result is not None:
                return result
    return None


def load_thresholds(root: Path) -> tuple[np.ndarray, str]:
    candidates = []
    for pattern in (
        "*threshold*.json",
        "*threshold*.npy",
        "metrics.json",
        "evaluation*.json",
        "*.yaml",
        "*.yml",
    ):
        candidates.extend(root.rglob(pattern))

    seen = set()
    for path in sorted(candidates):
        if path in seen or not path.is_file():
            continue
        seen.add(path)

        try:
            if path.suffix == ".npy":
                vector = np.asarray(
                    np.load(path), dtype=float
                ).reshape(-1)
            elif path.suffix in {".yaml", ".yml"}:
                with path.open("r", encoding="utf-8") as handle:
                    vector = extract_threshold_vector(
                        yaml.safe_load(handle)
                    )
            else:
                with path.open("r", encoding="utf-8") as handle:
                    vector = extract_threshold_vector(
                        json.load(handle)
                    )
        except Exception:
            continue

        if (
            vector is not None
            and vector.shape == (len(LABELS),)
            and np.all((0 <= vector) & (vector <= 1))
        ):
            return vector, str(path)

    warnings.warn(
        "No seven-value threshold vector was found; using 0.5."
    )
    return np.full(len(LABELS), 0.5), "fallback 0.5"


def model_device(model) -> torch.device:
    try:
        return model.get_input_embeddings().weight.device
    except Exception:
        return next(model.parameters()).device


def predict_pairs(
    model,
    tokenizer,
    frame: pd.DataFrame,
    thresholds: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    if tokenizer.pad_token_id is None:
        if tokenizer.eos_token_id is None:
            raise ValueError("Tokenizer has no pad or EOS token.")
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left" if IS_QLORA else "right"
    model.config.pad_token_id = tokenizer.pad_token_id
    model.eval()

    batch_size = 1 if IS_QLORA else 16
    probabilities = []
    device = model_device(model)

    for start in range(0, len(frame), batch_size):
        batch = frame.iloc[start:start + batch_size]
        encoded = tokenizer(
            batch["source_text"].astype(str).tolist(),
            text_pair=batch["simplified_text"].astype(str).tolist(),
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        with torch.inference_mode():
            logits = model(**encoded).logits
            probs = torch.sigmoid(logits.float()).cpu().numpy()

        probabilities.append(probs)

    probabilities = np.concatenate(probabilities, axis=0)
    predictions = (
        probabilities >= thresholds.reshape(1, -1)
    ).astype(int)
    return probabilities, predictions


def classify_pair(
    source_text: str,
    simplified_text: str,
) -> pd.DataFrame:
    frame = pd.DataFrame(
        {
            "source_text": [source_text],
            "simplified_text": [simplified_text],
        }
    )
    probabilities, predictions = predict_pairs(
        model, tokenizer, frame, thresholds
    )
    return pd.DataFrame(
        {
            "label": LABELS,
            "probability": probabilities[0],
            "threshold": thresholds,
            "predicted": predictions[0],
        }
    )


def release_memory() -> None:
    global model, tokenizer
    try:
        del model
        del tokenizer
    except NameError:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## Download the public repository and load the model

In [ ]:
root = repo_snapshot()
model, tokenizer, loading_mode = load_model_from_public_repo(root)
thresholds, threshold_source = load_thresholds(root)

print("Loading mode:", loading_mode)
print("Threshold source:", threshold_source)
print("Thresholds:", dict(zip(LABELS, thresholds)))

## Classify one original sentence and its Easy-to-Read rewrite

In [ ]:
result = classify_pair(
    source_text=(
        "The committee postponed the meeting because several members "
        "were absent."
    ),
    simplified_text=(
        "Several members were not there, so the committee moved the "
        "meeting to another day."
    ),
)

print(
    "Predicted strategies:",
    result.loc[result["predicted"] == 1, "label"].tolist(),
)
result

## Classify a CSV file

The file must contain `source_text` and `simplified_text` columns.

In [ ]:
# Google Colab upload:
try:
    from google.colab import files
    uploaded = files.upload()
    csv_path = next(iter(uploaded))
except ImportError:
    # Local Jupyter: replace this with your file path.
    csv_path = "example_pairs.csv"

pairs = pd.read_csv(csv_path)
required = {"source_text", "simplified_text"}
missing = required - set(pairs.columns)
if missing:
    raise KeyError(f"Missing CSV columns: {sorted(missing)}")

probabilities, predictions = predict_pairs(
    model, tokenizer, pairs, thresholds
)

output = pairs.copy()
for index, label in enumerate(LABELS):
    output[f"prob_{label}"] = probabilities[:, index]
    output[f"pred_{label}"] = predictions[:, index]

output["predicted_labels"] = [
    "; ".join(
        label
        for label, value in zip(LABELS, row)
        if value == 1
    )
    for row in predictions
]

output_path = f"{REPO_ID.split('/')[-1]}_predictions.csv"
output.to_csv(output_path, index=False)
print("Saved:", output_path)
output.head()

## Release memory

In [ ]:
release_memory()